# ZEREOS · ACE-Step 1.5 Arabic Singing Engine

This notebook launches a temporary GPU API for ZEREOS. Choose Runtime → Change runtime type → T4 GPU if Colab offers it, then run all cells. Free GPU access and session duration are not guaranteed.

The engine is open-source (MIT). The notebook creates a random API token and temporary HTTPS URL. Do not share them. Keep this notebook running while generating songs.


In [ ]:
import torch, os, subprocess, secrets, time, re, requests
if not torch.cuda.is_available():
    raise RuntimeError('لا توجد GPU متاحة. اختر T4 GPU إن كانت متاحة ثم أعد التشغيل.')
print('GPU:', torch.cuda.get_device_name(0))
WORK='/content/ACE-Step-1.5'
if not os.path.isdir(WORK): subprocess.run(['git','clone','--depth','1','https://github.com/ace-step/ACE-Step-1.5.git',WORK],check=True)
os.chdir(WORK)
print('Repository ready:',WORK)


In [ ]:
# Install upstream dependencies; this may take several minutes.
subprocess.run(['pip','install','uv'],check=True)
subprocess.run(['uv','sync'],check=True)
print('ACE-Step dependencies installed.')


In [ ]:
# Allow the ZEREOS GitHub Pages origin. Upstream defaults only allow localhost.
route_file=os.path.join(WORK,'acestep','api','route_setup.py')
src=open(route_file,encoding='utf-8').read()
old='allow_origins=["null", "http://localhost", "http://127.0.0.1"],'
new='allow_origins=["null", "http://localhost", "http://127.0.0.1", "https://arkanws513-spec.github.io"],'
if old in src: open(route_file,'w',encoding='utf-8').write(src.replace(old,new,1))
elif 'https://arkanws513-spec.github.io' not in src: raise RuntimeError('CORS patch did not match upstream source; stopped safely.')
print('ZEREOS origin is allowed for this session.')


In [ ]:
# Launch API with a random, session-only access token and a small language model.
os.chdir(WORK)
token=secrets.token_urlsafe(32)
env=os.environ.copy(); env['ACESTEP_API_KEY']=token; env['ACESTEP_CONFIG_PATH']='acestep-v15-turbo'; env['ACESTEP_LM_MODEL_PATH']='acestep-5Hz-lm-0.6B'; env['ACESTEP_INIT_LLM']='true'
log=open('/content/zereos-acestep-api.log','w')
api=subprocess.Popen(['uv','run','acestep-api','--host','0.0.0.0','--port','8001','--api-key',token,'--init-llm','--lm-model-path','acestep-5Hz-lm-0.6B'],cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
ready=False
for _ in range(180):
    if api.poll() is not None: break
    try:
        h=requests.get('http://127.0.0.1:8001/health',timeout=4)
        if h.ok: ready=True; print('API health:',h.json()); break
    except Exception: pass
    time.sleep(5)
if not ready:
    log.flush(); print(open('/content/zereos-acestep-api.log').read()[-9000:]); raise RuntimeError('ACE-Step API failed to start; inspect the log.')
print('SESSION TOKEN (copy privately into ZEREOS):',token)


In [ ]:
# Expose the temporary API through HTTPS. The tunnel expires with this Colab runtime.
deb='/content/cloudflared-linux-amd64.deb'
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb','-O',deb],check=True)
subprocess.run(['dpkg','-i',deb],check=True)
tlog=open('/content/zereos-acestep-tunnel.log','w')
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8001','--no-autoupdate'],stdout=tlog,stderr=subprocess.STDOUT)
public_url=None
for _ in range(60):
    time.sleep(2); tlog.flush(); match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',open('/content/zereos-acestep-tunnel.log').read())
    if match: public_url=match.group(0); break
if not public_url:
    print(open('/content/zereos-acestep-tunnel.log').read()[-5000:]); raise RuntimeError('Cloudflare temporary tunnel did not start.')
print('ENGINE URL (paste into ZEREOS):',public_url)
print('SESSION TOKEN (paste into ZEREOS):',token)
print('Open ZEREOS Music Studio: https://arkanws513-spec.github.io/Zereos-AI-Shorts/music-studio.html')
print('Keep this Colab runtime running. Do not share the URL or token.')


## Notes
- If Google does not grant a free GPU, generation cannot run in this session. Do not purchase anything to use this notebook.
- The first run downloads model weights and can take time.
- Inspect `/content/zereos-acestep-api.log` if the API fails.
- Stop the runtime when finished. The URL and token expire with the session.
